# Merge for Conservative Sample of Elliptical Galaxies

The aim of this section is to merge the data from the categories:

* samiDR3VisualMorphology.csv             *(to get galaxy type)*
* sameDR3InputCatClusters.csv             *(to get galaxy luminosity/mass)*
* samiDR3StelKin.csv                      *(to get the galaxy velocity dispersion)*

From the merged data set only elliptical galaxies (type == 0) will be selected.

In [1]:
import pandas as pd

In [2]:
# access the SAMI visual morphology
samimorph = pd.read_csv('SAMI_Visual_Morphology/samiDR3VisualMorphology.csv')
samicat = pd.read_csv('SAMI_Input_Cat_Cluster/samiDR3InputCatClusters.csv')
samikin = pd.read_csv('SAMI_Kinematics/samiDR3StelKin.csv')

print('Sample Size of SAMI Visual Morphology Dataset:', len(samimorph))
print('Sample Size of SAMI Input Catalogue Dataset:', len(samicat))
print('Sample Size of SAMI Stellar Kinematics Dataset:', len(samikin))

Sample Size of SAMI Visual Morphology Dataset: 3068
Sample Size of SAMI Input Catalogue Dataset: 1433
Sample Size of SAMI Stellar Kinematics Dataset: 3426


In [4]:
# merging the three datasets into one data frame using the catid as the common variable

morphcat = pd.merge(samimorph, samicat, on='catid', how='inner')
df = pd.merge(morphcat, samikin, on='catid', how='inner')
print('Final Data Set', len(df))
print(df.head(20))

# Comments on output:
    # opening df a scrollable element reveals that there is quite a few NaN values in the table which
    # can potentially have an effect on our Faber-Jackson relation. These will need to be filtered out.

Final Data Set 962
    Unnamed: 0_x       catid  type  Unnamed: 0_y     ra_obj   dec_obj  \
0           2151  9008500001   0.0           849  10.460211 -9.303184   
1           2152  9008500002   0.0           850  10.460158 -9.303645   
2           2153  9008500004   1.5           851  10.456779 -9.295304   
3           2154  9008500007   0.0           852  10.450953 -9.284216   
4           2154  9008500007   0.0           852  10.450953 -9.284216   
5           2155  9008500018   0.5           853  10.489803 -9.281996   
6           2156  9008500020   0.0           854  10.422118 -9.315872   
7           2157  9008500022   0.5           855  10.436680 -9.338163   
8           2158  9008500024   0.5           856  10.417239 -9.304267   
9           2159  9008500034   1.0           857  10.470191 -9.358393   
10          2160  9008500042   0.0           858  10.400896 -9.325084   
11          2161  9008500043   3.0           859  10.426972 -9.357179   
12          2162  9008500059   0

In [5]:
elliptical = df[df['type'] == 0.0] # choosing only elliptical galaxies
# key defines elliptical galaxies as having the type = 0
    # 0.5 is for galaxies between elliptical and lenticular stages

print('Number of Elliptical Galaxies:', len(elliptical))
print('Elliptical Galaxies:', elliptical.head(20))

Number of Elliptical Galaxies: 152
Elliptical Galaxies:     Unnamed: 0_x       catid  type  Unnamed: 0_y     ra_obj   dec_obj  \
0           2151  9008500001   0.0           849  10.460211 -9.303184   
1           2152  9008500002   0.0           850  10.460158 -9.303645   
3           2154  9008500007   0.0           852  10.450953 -9.284216   
4           2154  9008500007   0.0           852  10.450953 -9.284216   
6           2156  9008500020   0.0           854  10.422118 -9.315872   
10          2160  9008500042   0.0           858  10.400896 -9.325084   
12          2162  9008500059   0.0           860  10.394659 -9.350048   
14          2164  9008500064   0.0           862  10.484774 -9.382390   
16          2166  9008500068   0.0           864  10.395616 -9.360660   
20          2170  9008500077   0.0           868  10.554106 -9.297421   
27          2177  9008500096   0.0           876  10.370102 -9.232952   
34          2184  9008500114   0.0           883  10.459061 -9.42984

# Cleaning Data

Extracting data columns:
* Effective Brightness
* Velocity Dispersion at Effective Radius

Filtering out NaN values

In [10]:
# making data frame with only the effective brightness and velocity dispersion at one effective radius

rawdf = elliptical[['mu_1re', 'sigma_re']]
print('Uncleaned Data Set:', len(rawdf))
print(rawdf.head(20))



Uncleaned Data Set: 152
       mu_1re    sigma_re
0   22.194613         NaN
1   22.220806         NaN
3   20.561842  265.057370
4   20.561842         NaN
6   22.480024  170.477860
10  20.710413  202.322860
12  22.171244  204.846340
14  21.802141  125.197975
16  21.571623   87.443590
20  22.191960  183.164600
27  22.008698  164.250000
34  22.396560         NaN
36  22.136293  122.640700
44  20.584938  114.042900
45  21.034206  131.962300
47  24.789797         NaN
52  21.824500  173.848070
57  21.654446  166.522140
59  24.615583         NaN
71  21.080845  232.152270


In [13]:
NaNbright = len(elliptical[elliptical['mu_1re'].isnull()])
print('Number of NaN values in Brightness at one Effective Radius:', NaNbright)
# how many values are NaN in the effective brightness column

NaNvelocity = len(elliptical[elliptical['sigma_re'].isnull()])
print('Number of NaN values in Velocity Dispersion at one Effective Radius:', NaNvelocity)
# how many values are NaN in the effective velocity dispersion column

NaNtotal = NaNbright + NaNvelocity
print('Total NaN values:', NaNtotal)

Number of NaN values in Brightness at one Effective Radius: 1
Number of NaN values in Velocity Dispersion at one Effective Radius: 45
Total NaN values: 46


In [15]:
# Filtering out NaN values
   # reference code: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html

df_clean = df.dropna()

print('Cleaned Data Set', len(df_clean))
print(df_clean.head(20))

RemovedSamples = len(df) - len(df_clean)
print('Number of Samples Removed:', RemovedSamples)

print('Does Removed Samples = Total NaN values?', RemovedSamples == NaNtotal)

Cleaned Data Set 107
        mu_1re    sigma_re
3    20.561842  265.057370
6    22.480024  170.477860
10   20.710413  202.322860
12   22.171244  204.846340
14   21.802141  125.197975
16   21.571623   87.443590
20   22.191960  183.164600
27   22.008698  164.250000
36   22.136293  122.640700
44   20.584938  114.042900
45   21.034206  131.962300
52   21.824500  173.848070
57   21.654446  166.522140
71   21.080845  232.152270
74   22.053015  213.418180
92   22.070808  187.230400
93   21.813902   96.251780
99   21.399350  193.565280
100  21.056297  197.126070
102  21.211555  105.533780
Number of Samples Removed: 45
Does Removed Samples = Total NaN values? False


In [ ]:
# Filtering out NaN values in the vsigma columns
    # reference code: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html

# df_clean = df.dropna(subset=['vsigma_re_mge', 'vsigma_re_mge_err', 'vsigma_max_mge', 'vsigma_max_mge_err'])
    # removing galaxies with NaN values in vsigma columns)

# print('Cleaned Data Set', len(df_clean))
# print(df_clean.head(20))